# GVP-EGNN v2.1 — SIIMPL Crystalline-Target Training

End-to-end Colab notebook for retraining the v2 GVP-EGNN + NSF flow on SIIMPL data (replaces v2's SRIM amorphous-target training).

**What this trains:**
- Same architecture as v2 (~4.1 M parameters, GVP-EGNN backbone + NSF flow + vMF/Gaussian aux heads).
- Same two-stage protocol (Stage 1 joint, Stage 2 flow-only fine-tune).
- **Different:** Oh-group augmentation (preserves channeling), config-aware train/val split (prevents leakage), Stage 2 on Pool A only (keeps posterior prior uniform-on-S²).

**Required Google Drive layout** (mounted at `/content/drive/MyDrive`):
```
sbi-srim-results/
├── inverse-ml-repo.zip            ← zip of the Inverse ML repo (or clone from git)
├── siimpl_data/
│   ├── siimpl_train.csv            ← 1.9 GB, from local generation
│   └── siimpl_eval.csv             ← 363 MB
└── gvp_egnn_v21_siimpl/           ← will be created for checkpoints (auto-saved)
```

**Estimated wall time on A100-40GB:** ~25–35 hours (matches v2 SRIM ballpark, slightly longer due to 25% more data).

## 1. Mount Drive, install dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install pinned packages used by the training pipeline
!pip install -q sbi==0.22.0 nflows==0.14 pyknos==0.16 scipy pandas scikit-learn matplotlib seaborn tensorboard
!nvidia-smi

## 2. Unpack the code from Drive

The v2.1 code lives in `inverse-ml-v21.zip` on your Drive — a clean snapshot of branch `siimpl/v21`. The cell below unzips it into `/content/inverse-ml/` and sets that as the working directory. No GitHub authentication required.

**One-time setup before running this notebook:** upload the zip (16 MB) to your Drive at:
```
/MyDrive/sbi-srim-results/inverse-ml-v21.zip
```

If you also haven't uploaded the SIIMPL data CSVs yet, do it now too (see section 3).

In [ ]:
import os, pathlib

ZIP_PATH = '/content/drive/MyDrive/sbi-srim-results/inverse-ml-v21.zip'
TARGET   = '/content/inverse-ml'

if not pathlib.Path(ZIP_PATH).exists():
    raise FileNotFoundError(
        f'Zip not found at {ZIP_PATH}. Upload inverse-ml-v21.zip to Drive first.')

if not os.path.exists(TARGET):
    !unzip -q {ZIP_PATH} -d /content/
    print(f'Unzipped to {TARGET}')
else:
    print(f'Already present at {TARGET}')

%cd /content/inverse-ml
!ls src/scripts/train_gvp_egnn_v21.py && echo '  v21 training script present'

## 3. Link SIIMPL data from Drive into the repo's expected path

The script expects `data/siimpl/siimpl_train.csv` and `data/siimpl/siimpl_eval.csv`. Symlink (or copy) them so the training script finds them.

In [ ]:
import os, pathlib

DATA_SRC = '/content/drive/MyDrive/sbi-srim-results/siimpl_data'
DATA_DST = pathlib.Path.cwd() / 'data' / 'siimpl'
DATA_DST.mkdir(parents=True, exist_ok=True)

for fname in ('siimpl_train.csv', 'siimpl_eval.csv'):
    src = pathlib.Path(DATA_SRC) / fname
    dst = DATA_DST / fname
    if not src.exists():
        raise FileNotFoundError(f'Source missing: {src}. Upload from local first.')
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    # Symlink (fast); fall back to copy if symlinks not supported
    try:
        dst.symlink_to(src)
    except OSError:
        import shutil; shutil.copy(src, dst)
    sz_mb = src.stat().st_size / 1e6
    print(f'  {fname}: {sz_mb:.0f} MB')

print('\nData ready at', DATA_DST)
!ls -lh {DATA_DST}

## 4. Sanity-check data + model load

Quick check that the script can import + reach the data, before kicking off a 30-hour run.

In [ ]:
import sys, importlib.util, numpy as np, torch
sys.path.insert(0, '.')

spec = importlib.util.spec_from_file_location(
    'train_v21', 'src/scripts/train_gvp_egnn_v21.py')
mod = importlib.util.module_from_spec(spec)
sys.modules['train_v21'] = mod
spec.loader.exec_module(mod)

print('TRAIN_CSV:', mod.TRAIN_CSV, '— exists:', mod.TRAIN_CSV.exists())
print('EVAL_CSV:', mod.EVAL_CSV, '— exists:', mod.EVAL_CSV.exists())
print('RESULTS_DIR:', mod.RESULTS_DIR)
print('POOL_B_ION_START:', mod.POOL_B_ION_START)
print('MAX_TRAIN_TRACKS:', mod.MAX_TRAIN_TRACKS)

# Oh-group integrity
dets = np.linalg.det(mod.OH_GROUP_NP)
n_proper = int((dets > 0).sum()); n_improper = int((dets < 0).sum())
print(f'Oh group: {n_proper} proper + {n_improper} improper = {n_proper+n_improper}')
assert (n_proper, n_improper) == (24, 24)

# Aug preserves unit norm
B, n_max, k = 4, 10, 8
x = torch.randn(B, n_max * (3 + k))
th = torch.cat([torch.rand(B, 1) * 100, torch.nn.functional.normalize(torch.randn(B, 3), dim=-1)], dim=-1)
_, th_o = mod.apply_oh_augmentation(x.clone(), th.clone(), n_max)
norms = torch.linalg.norm(th_o[:, 1:4], dim=-1)
assert torch.allclose(norms, torch.ones_like(norms), atol=1e-5), norms
print('Direction-vector norms after Oh aug:', norms.tolist())
print('ALL CHECKS PASS')

## 5. Smoke test (≈ 5 minutes)

Runs Stage 1 for 5 epochs on 5,000 tracks. Confirms the end-to-end pipeline works (preprocessing, kNN, forward/backward, Oh aug, loss computation) before the full run.

In [ ]:
!python -m src.scripts.train_gvp_egnn_v21 --smoke 2>&1 | tail -100

## 6. Full training

**Stage 1 (joint training):** 200 epochs max, early stop at 30. ~25 hr on A100-40GB.

**Stage 2 (flow-only, Pool A only):** 60 epochs max, early stop at 20. ~3 hr.

Auto-saves checkpoints + log to `/content/drive/MyDrive/sbi-srim-results/gvp_egnn/` every 10 epochs (crash-safe). Loss curves stream to TensorBoard under `results/gvp_egnn_v21_siimpl/tb_logs/`.

**Note:** the script's built-in eval suite (SBC, TARP, ECE, geometric baselines, single-track timing benchmark) runs automatically after Stage 2 completes — no extra cell needed.

In [ ]:
# Launch full training — run this and let it complete.
# Output streams live; rerun with --resume if the runtime disconnects.
!python -m src.scripts.train_gvp_egnn_v21 2>&1 | tee training_log.txt

## 7. Persist results to Drive

The script auto-saves every 10 epochs to Drive, but copy final artifacts explicitly at the end.

In [ ]:
import shutil, pathlib

src = pathlib.Path('results/gvp_egnn_v21_siimpl')
dst = pathlib.Path('/content/drive/MyDrive/sbi-srim-results/gvp_egnn_v21_siimpl')
dst.mkdir(parents=True, exist_ok=True)

for item in src.iterdir():
    target = dst / item.name
    if item.is_file():
        shutil.copy2(item, target)
    else:
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(item, target)

print('Synced to:', dst)
!ls -lh {dst}

## 8. Inspect key results

Quick textual summary of the training outcome. Plots (SBC ranks, TARP, posterior corners) saved to `results/gvp_egnn_v21_siimpl/`.

In [ ]:
import pandas as pd, pathlib

log_path = pathlib.Path('results/gvp_egnn_v21_siimpl/training_log.csv')
eval_path = pathlib.Path('results/gvp_egnn_v21_siimpl/eval_results.csv')

if log_path.exists():
    log = pd.read_csv(log_path)
    print(f'Stage 1 epochs run: {len(log)}')
    print(f'Best val_flow:     {log["val_flow"].min():.4f} (epoch {log["val_flow"].idxmin()})')
    print(f'Final epoch_sec:   {log["epoch_sec"].iloc[-1]:.1f} s')
    print(f'Total wall:        {log["elapsed_min"].iloc[-1]:.1f} min')
    print()

if eval_path.exists():
    df = pd.read_csv(eval_path)
    print(f'Eval tracks: {len(df):,}')
    print(f'Median axis error:   {df["angular_error_deg"].median():.2f}°')
    print(f'Head-tail accuracy:  {(df["angular_error_deg"] <= 90).mean()*100:.1f}%')
    print(f'Median energy error: {df["energy_error"].median():.2f} keV')
    if 'angular_cone_68' in df.columns:
        print(f'Median 68% cone:     {df["angular_cone_68"].median():.2f}°')
    print()
    print('Angular error percentiles:')
    print(df['angular_error_deg'].quantile([0.25, 0.5, 0.75, 0.9]).to_string())

In [ ]:
# Show SBC rank plot + TARP plot inline
from IPython.display import Image, display
for name in ('eval_sbc_ranks.png', 'eval_tarp.png'):
    p = pathlib.Path(f'results/gvp_egnn_v21_siimpl/{name}')
    if p.exists():
        print(name)
        display(Image(str(p)))